# 2.2 — Procesamiento: Monitor de Sequía (quincenal → mensual)

**Capa 3 del protocolo.** Toma los registros tidy de
`01_ingesta/1.2-monitor-sequia.ipynb` (municipio × fecha × categoría) y los
lleva a una serie mensual para Hermosillo.

## Entradas

| Ruta | Contenido |
|------|-----------|
| `data/interim/sonora_sequia/sonora_sequia_AAAA.csv` | Monitor de Sequía, municipios de Sonora: `CVE_MUN`, `fecha`, `categoria_sequia`, `severidad_num` |

## Salidas

- `data/interim/sequia/sequia_hermosillo_mensual.csv` (opcional, para
  revisión): `fecha_mes`, `severidad_sequia` (media 0–5), `categoria_sequia`
  (moda del mes).

> No guardar nada en `data/interim/sonora_sequia/`: `panel.py` lee **todos**
> los CSV de esa carpeta.

## Checklist

- [ ] Registros de Hermosillo (`CVE_MUN == 30`) en la ventana del panel.
- [ ] Registros por mes (1 o 2 quincenas) y si la frecuencia cambia entre años.
- [ ] Categorías sin mapeo en `config.DROUGHT_SEVERITY_MAP` (la ingesta las deja en 0).
- [ ] Moda del mes cuando las quincenas tienen categorías distintas (desempate).
- [ ] Meses del panel sin registro.

## Configuración e imports

In [ ]:
from pathlib import Path
import sys

ROOT_DIR = Path.cwd().parent.parent  # notebooks/<seccion>/ -> raiz del proyecto
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

import pandas as pd
import matplotlib.pyplot as plt

from sonora_health_climate_eda.config import (
    DROUGHT_EXTREME_MIN_SEVERITY,
    DROUGHT_SEVERITY_MAP,
    INTERIM_DATA_DIR,
    PANEL_END_YEAR,
    PANEL_START_YEAR,
)
from sonora_health_climate_eda.panel import (
    _drought_records_hermosillo,  # privada: registros quincenales de Hermosillo
    drought_hermosillo,
)

OUT_DIR = INTERIM_DATA_DIR / "sequia"  # no usar sonora_sequia/ (ver nota arriba)

## Registros quincenales de Hermosillo

In [ ]:
registros = _drought_records_hermosillo()
registros = registros[registros["fecha"].dt.year.between(PANEL_START_YEAR, PANEL_END_YEAR)].copy()
registros["fecha_mes"] = registros["fecha"].dt.to_period("M").dt.to_timestamp()
print(
    f"{len(registros)} registros, "
    f"{registros['fecha'].min():%Y-%m-%d} a {registros['fecha'].max():%Y-%m-%d}"
)
registros[["fecha", "categoria_sequia", "severidad_num"]].head()

## Frecuencia de la fuente

In [ ]:
por_mes = registros.groupby("fecha_mes").size()
print("Meses según número de registros:")
print(por_mes.value_counts().sort_index())
print("\nRegistros por año:")
print(registros.groupby(registros["fecha"].dt.year).size())

## Categorías y mapeo a severidad

In [ ]:
print(registros["categoria_sequia"].value_counts(dropna=False))

no_mapeadas = sorted(set(registros["categoria_sequia"].dropna().astype(str)) - set(DROUGHT_SEVERITY_MAP))
print("\nCategorías sin mapeo (la ingesta las convierte en severidad 0):", no_mapeadas)
print("Registros con categoría vacía:", registros["categoria_sequia"].isna().sum())

## Serie mensual

In [ ]:
sequia_mensual = drought_hermosillo()

esperados = pd.date_range(f"{PANEL_START_YEAR}-01-01", f"{PANEL_END_YEAR}-12-01", freq="MS")
faltantes = esperados.difference(sequia_mensual["fecha_mes"])
print("Meses del panel sin registro de sequía:", list(faltantes.strftime("%Y-%m")))
sequia_mensual.head()

## Moda con empate

Si un mes tiene dos quincenas con categorías distintas, la moda queda
empatada y `drought_hermosillo()` toma una sin un criterio explícito.

In [ ]:
cats_mes = registros.groupby("fecha_mes")["categoria_sequia"].nunique()
meses_mixtos = cats_mes[cats_mes > 1].index
print(f"Meses con más de una categoría: {len(meses_mixtos)}")
registros[registros["fecha_mes"].isin(meses_mixtos)][["fecha", "categoria_sequia", "severidad_num"]]

# TODO: definir el desempate (p. ej. quedarse con la categoría más severa del mes)
# e implementarlo en panel.drought_hermosillo():
# sev_max = registros.groupby("fecha_mes")["severidad_num"].max()
# sequia_mensual["severidad_max_mes"] = sequia_mensual["fecha_mes"].map(sev_max)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 3.5))
ax.step(sequia_mensual["fecha_mes"], sequia_mensual["severidad_sequia"], where="post")
ax.axhline(DROUGHT_EXTREME_MIN_SEVERITY, ls="--", color="gray", label="umbral de mes_sequia (≥ D1)")
ax.set_yticks(range(6), ["Sin sequía", "D0", "D1", "D2", "D3", "D4"])
ax.set_title("Severidad de sequía mensual — Hermosillo")
ax.legend()
plt.tight_layout()
plt.show()

## Guardar salida

In [ ]:
# TODO: guardar una vez validada la serie
# OUT_DIR.mkdir(parents=True, exist_ok=True)
# sequia_mensual.to_csv(OUT_DIR / "sequia_hermosillo_mensual.csv", index=False)

## Decisiones de procesamiento

Registrar cada decisión que cambie los datos y su justificación. Si la
decisión cambia el panel, implementarla en `sonora_health_climate_eda/panel.py`
(es lo que ejecuta `03_eda/3.0-integracion-espacial.ipynb`) y llevarla al
reporte final.

| Variable / periodo | Problema | Decisión | Justificación |
|--------------------|----------|----------|---------------|
| Quincenal → mensual | Dos registros por mes | Severidad promedio + categoría modal | Se pierde el detalle de eventos cortos (limitación documentada) |
| _TODO_ | | | |